# 03 - Churn Analysis & Business Recommendations
Translate analysis into actionable business strategies and estimate potential revenue impact.

## 1. Load Data

In [1]:
import sqlite3 
import pandas as pd

In [ ]:
conn = sqlite3.connect("db/telecom.db")

df = pd.read_sql_query("SELECT * FROM telecom_features", conn)

## 2. Revenue At Risk 

MonthlyCharges is primarily used as a proxy for monthly recurring revenue (MRR).

Total estimated revenue at risk from churned customers is approximately $139K per month.

## 3. Prioritization / Scoring

Segments are based on a simple risk score combining:
- churn rate
- segment rate
- average monthly charges

This helps identify where retention efforts can have the greatest impact.

In [3]:
segment = pd.read_sql_query("""
SELECT
  Contract,
  tenure_group,
  COUNT(*) AS customers,
  AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END) AS churn_rate,
  AVG(MonthlyCharges) AS avg_monthly_charges,
  SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) AS revenue_at_risk_mrr
FROM telecom_features
GROUP BY Contract, tenure_group
ORDER BY revenue_at_risk_mrr DESC
""", conn)

segment.head()

,Contract,tenure_group,customers,churn_rate,avg_monthly_charges,revenue_at_risk_mrr
0,Month-to-month,0-6,1413,0.552017,55.877141,49681.30
1,Month-to-month,25+,1144,0.308566,78.781906,30565.35
2,Month-to-month,13-24,737,0.377205,69.309566,21980.30
3,Month-to-month,7-12,581,0.419966,63.910671,18620.15
4,One year,25+,1152,0.118924,71.646354,12364.30


In [4]:
segment['risk_score'] =(
    segment['churn_rate'] *
    segment['customers'] *
    segment['avg_monthly_charges']
)
segment.sort_values(by='risk_score', ascending=False).head(10)

,Contract,tenure_group,customers,churn_rate,avg_monthly_charges,revenue_at_risk_mrr,risk_score
0,Month-to-month,0-6,1413,0.552017,55.877141,49681.30,43584.169851
1,Month-to-month,25+,1144,0.308566,78.781906,30565.35,27810.012675
2,Month-to-month,13-24,737,0.377205,69.309566,21980.30,19268.059294
3,Month-to-month,7-12,581,0.419966,63.910671,18620.15,15594.203787
4,One year,25+,1152,0.118924,71.646354,12364.30,9815.550521
5,Two year,25+,1537,0.031230,63.756604,4165.30,3060.316981
6,One year,13-24,197,0.081218,44.878680,1101.35,718.058883
7,One year,7-12,85,0.105882,39.672353,438.00,357.051176
8,One year,0-6,39,0.102564,27.352564,214.80,109.410256
9,Two year,0-6,29,0.000000,36.096552,0.00,0.000000


Higher risk scores indicate segments with both high churn and high revenue impact.

## 4. Recommendations

**Target Early-Tenure Customers**
- Focus onboarding and engagement efforts on customers in their first 6-12 months

**Encourage Long-Term Contracts**
- Provide incentives for customers to move from month-to-month annual plans

**Improve Value for High-Paying Customers**
- Bundle services or adjust pricing to reduce churn among higher-paying users

**Promote Auto Pay Adoption**
- Encourage automatic payment methods to reduce friction and improve retention


## 5. Scenario Analysis
To estimate business impact, we model simple improvements in churn reduction.

In [5]:
revenue_at_risk = 139130

Monthly revenue at risk from churned customers is approximately $139K.

In [6]:
# Scenario 1: 5% churn reduction
saved_rev = revenue_at_risk * 0.05

# Scenario 2: 10% improvement in high-risk segments (~40% of churn)
saved_targeted = revenue_at_risk * 0.4 * 0.10

# Scenario 3: contract conversion impact (~8% improvement)
saved_contract = revenue_at_risk * 0.08

saved_rev, saved_targeted, saved_contract

(6956.5, 5565.200000000001, 11130.4)

- A 5% reduction in churn could retain approximately $7K in monthly revenue (~$83K annually)

- Targeting high-risk could retain ~$5-6K monthly

- Increasing long-term contract adoption could retain ~10K+ monthly

## 6. Executive Summary

### Key Findings:

- Churn is highest among early tenure customers

- Month-to-month contracts show significantly higher churn

- Revenue risk is concentrated in a small number of segments

- Higher monthly charges are associated with increased churn

### Recommended Actions:

- Improve onboarding for new customers

- Incentivize long-term contract adoption

- Address pricing and value perception for high-paying customers 

- Promote auto pay enrollment

### Measurement Plan:

- Track churn by tenure group

- Monitor contract conversion rates

- Measure retained monthly revenue from targeted interventions